# Работа с LLM через API: GigaChat + LangChain

Практическое занятие 2. Курс «Введение в профессию».

Задача: автоматически извлекать из текстовых заявок на аренду жилья количество проживающих, используя модель GigaChat, подключённую через `langchain-gigachat`.

## 1. Импортируем компоненты LangChain

- `GigaChat` — интерфейс модели;
- `SystemMessagePromptTemplate`, `HumanMessagePromptTemplate` — шаблоны системного и пользовательского сообщений;
- `PromptTemplate`, `ChatPromptTemplate` — сборка промптов;
- `StrOutputParser` — парсинг ответа как строки;
- `os` и `dotenv` — безопасное чтение ключа из `.env`;
- `pandas` — работа с таблицей заявок.

In [ ]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

# Ключ читается из .env (переменная GIGA_KEY), а не хранится в коде
load_dotenv()
api_key = os.getenv("GIGA_KEY")
if not api_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

## 2. Инициализация модели и проверка подключения

In [ ]:
llm = GigaChat(
    credentials=api_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

response = llm.invoke("Привет! Как дела?")
print(response.content)

## 3. Системное и пользовательское промптирование

Системное сообщение задаёт роль модели, пользовательское — конкретную заявку и инструкцию.

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья."
)
user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\nИзвлеки количество человек и верни только число."
)
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

chain = chat_prompt | llm | StrOutputParser()
print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))

## 4. Базовый промпт

Простой `PromptTemplate` + `StrOutputParser`: модель возвращает только число.

In [ ]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain_basic = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]
for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain_basic.invoke({'text': text})}")
    print("---")

## 5. Загружаем 15 заявок варианта 19

Файл `rental_19.csv` содержит столбец `text` (заявка) и `amount` (эталонное количество проживающих).

In [ ]:
df = pd.read_csv("rental_19.csv", sep=";", encoding="utf-8")
df.head()

## 6. Прогоняем каждую заявку через модель

Идём по столбцу `text`, собираем ответы и записываем их в новый столбец `result`, затем сохраняем таблицу в CSV.

In [ ]:
results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain_basic.invoke({"text": text})
    except Exception as e:  # noqa: BLE001
        result = f"ERROR: {e}"
    results.append(result)

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["amount", "result"]]

## 7. Оценка точности системы (ветка llm-improve-1)

Сравниваем эталон `amount` и ответ модели `result`. Ответы — строки, поэтому преобразуем их в числа через `pd.to_numeric(..., errors='coerce')`: некорректные значения станут `NaN` и не совпадут с эталоном.

In [ ]:
pred = pd.to_numeric(df["result"], errors="coerce")
correct = int((pred == df["amount"]).sum())
total = len(df)
print(f"Ошибок: {total - correct}")
print(f"Точность: {correct / total:.1%}")

### Сравнение техник промптирования

Проверим, как на тех же 15 заявках работают детализированные инструкции, few-shot и Chain-of-Thought.

In [ ]:
detailed_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Ты — эксперт по анализу текстов заявок на аренду жилья. Извлеки точное количество людей, которые будут проживать.
ИНСТРУКЦИИ:
1. Найди все упоминания количества людей (прямые и косвенные)
2. Учти правила:
- "семья с одним ребенком" = 3 человека
- "семья с двумя детьми" = 4 человека
- "пара", "двое", "молодая семья" = 2 человека
- "один", "студент", "индивидуально" = 1 человек
3. Числительные переведи в цифры
4. Верни ТОЛЬКО целое число без пояснений

Текст заявки: {text}
Количество проживающих:""",
)

few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Извлеки количество людей из текста заявки. Изучи примеры:
Заявка: "Семья из трех человек ищет двухкомнатную квартиру"
Количество: 3
Заявка: "Молодая пара снимет студию в центре города"
Количество: 2
Заявка: "Ищу однокомнатную квартиру для себя"
Количество: 1
Заявка: "Семья с одним ребенком рассматривает варианты жилья"
Количество: 3
Заявка: "Четверо студентов ищут четырехкомнатную квартиру"
Количество: 4

Теперь проанализируй эту заявку:
Заявка: {text}
Количество:""",
)

cot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Проанализируй заявку на аренду жилья и определи количество проживающих по шагам.
Заявка: {text}
ШАГ 1: Найди все упоминания людей
ШАГ 2: Определи значение каждого упоминания
ШАГ 3: Сложи общее количество
ШАГ 4: Верни итоговое число
Пошаговый анализ:
ШАГ 1:
ШАГ 2:
ШАГ 3:
ШАГ 4:""",
)

techniques = {
    "Базовый": basic_prompt | llm | StrOutputParser(),
    "Детальный": detailed_prompt | llm | StrOutputParser(),
    "Few-shot": few_shot_prompt | llm | StrOutputParser(),
    "Chain-of-Thought": cot_prompt | llm | StrOutputParser(),
}

rows = []
for name, chain_t in techniques.items():
    preds = []
    for text in df["text"]:
        try:
            out = chain_t.invoke({"text": text})
        except Exception as e:  # noqa: BLE001
            out = f"ERROR: {e}"
        m = re.search(r"\d+", str(out))
        preds.append(int(m.group()) if m else None)
    acc = sum(1 for p, a in zip(preds, df["amount"]) if p == a) / len(df)
    rows.append({"Техника": name, "Точность": acc})

comparison = pd.DataFrame(rows)
comparison["Точность"] = comparison["Точность"].map("{:.1%}".format)
comparison

### Вывод

- **Базовый** промпт: точность **93.3%** (14 из 15). Единственная ошибка — заявка «Снимем жилье в Лазаревском с 30.07 по 09.08.15 г. Нас 3 взрослых, 2 детей 5 лет и 3,9 лет»: модель вернула 7 вместо 5, запутавшись в числах дат.
- **Детальный** промпт с явными правилами: **100%** (15 из 15) — именно эту технику выбираем как улучшение.
- **Few-shot**: точность упала до **73.3%** (11 из 15).
- **Chain-of-Thought**: **6.7%** при наивном парсинге первого числа — модель печатает шаги, и первым числом оказывается «1» из «ШАГ 1». Это показывает, что пошаговый вывод требует отдельного парсинга итогового ответа.